In [ ]:
import pandas as pd
import numpy as np

In [ ]:
df = pd.read_csv("profiles.csv")
df.head()

In [4]:
def preprocess_and_engineer(df):
    """
    Cleans the raw per-match log and adds derived features.
    1. Type coercion & outlier capping
    2. Missing value strategy per column type
    3. Per-match feature engineering (ratios, composite scores)
    4. Position grouping (for downstream position-aware analysis)
    """
    df = df.copy()

    # TYPE COERCION
    numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()

    # [1, 95]
    df["minutes_played"] = df["minutes_played"].clip(1, 95)

    # xG: can't be negative
    df["total_xg"] = df["total_xg"].clip(lower=0)

    # MISSING VALUE STRATEGY
    rate_fill_zero = [
        "pass_completion",
        "avg_pass_length",
        "avg_carry_dist",
        "avg_xg_per_shot",
        "avg_shot_distance",
    ]
    df[rate_fill_zero] = df[rate_fill_zero].fillna(0)

    if "defensive_height" in df.columns:
        df["defensive_height"] = df.groupby("position")["defensive_height"].transform(
            lambda x: x.fillna(x.median())
        )
        df["defensive_height"] = df["defensive_height"].fillna(
            df["defensive_height"].median()
        )

    df[numeric_cols] = df[numeric_cols].fillna(0)

    # OUTLIER CAPPING to count columns
    skip_cap = {
        "player_id",
        "match_id",
        "minutes_played",
        "pass_completion",
        "avg_pass_length",
        "avg_carry_dist",
        "avg_xg_per_shot",
        "avg_shot_distance",
        "defensive_height",
    }
    for col in numeric_cols:
        if col in skip_cap:
            continue
        q1, q3 = df[col].quantile(0.01), df[col].quantile(0.99)
        df[col] = df[col].clip(q1, q3)

    # PER-MATCH FEATURE ENGINEERING

    # Passing quality
    df["pass_completion_u_pressure"] = np.where(
        df["under_pressure_passes"] > 0,
        1 - (df["under_pressure_passes"] / df["total_passes"].replace(0, np.nan)),
        np.nan,
    )
    df["progressive_pass_ratio"] = df["progressive_passes"] / df[
        "total_passes"
    ].replace(0, np.nan)
    df["chance_creation_rate"] = (df["key_passes"] + df["goal_assists"]) / df[
        "total_passes"
    ].replace(0, np.nan)
    df["long_ball_ratio"] = df["long_passes"] / df["total_passes"].replace(0, np.nan)
    df["cross_ratio"] = df["crosses"] / df["total_passes"].replace(0, np.nan)

    # Carrying quality
    df["progressive_carry_ratio"] = df["progressive_carries"] / df[
        "total_carries"
    ].replace(0, np.nan)
    df["carry_box_ratio"] = df["carries_into_box"] / df["total_carries"].replace(
        0, np.nan
    )

    # Shooting quality
    df["shot_accuracy"] = df["shots_on_target"] / df["total_shots"].replace(0, np.nan)
    df["xg_per_90"] = df["total_xg"] / (df["minutes_played"] / 90).replace(0, np.nan)
    df["xg_overperformance"] = df["goals"] - df["total_xg"]
    df["goal_conversion"] = df["goals"] / df["total_shots"].replace(0, np.nan)

    # Defensive quality
    df["defensive_actions_p90"] = df["total_defensive_actions"] / (
        df["minutes_played"] / 90
    ).replace(0, np.nan)
    df["tackle_success_proxy"] = df["tackles"] / df["total_duels"].replace(0, np.nan)
    df["pressure_regain_proxy"] = df["counterpresses"] / df["pressures"].replace(
        0, np.nan
    )

    # Dribbling quality
    df["dribble_success_rate"] = df["successful_dribbles"] / df[
        "total_dribbles"
    ].replace(0, np.nan)

    # Duel quality
    df["duel_win_rate"] = df["duels_won"] / df["total_duels"].replace(0, np.nan)

    # Possession discipline
    df["turnover_rate"] = df["turnovers"] / df["touches"].replace(0, np.nan)
    df["touches_per_minute"] = df["touches"] / df["minutes_played"].replace(0, np.nan)

    # POSITION GROUPING
    position_map = {
        # Goalkeeper
        "Goalkeeper": "GK",
        # Defenders
        "Center Back": "DEF",
        "Right Back": "DEF",
        "Left Back": "DEF",
        "Right Center Back": "DEF",
        "Left Center Back": "DEF",
        "Right Wing Back": "DEF",
        "Left Wing Back": "DEF",
        # Midfielders
        "Defensive Midfield": "MID",
        "Center Midfield": "MID",
        "Left Center Midfield": "MID",
        "Right Center Midfield": "MID",
        "Left Defensive Midfield": "MID",
        "Right Defensive Midfield": "MID",
        "Attacking Midfield": "MID",
        "Center Attacking Midfield": "MID",
        "Left Attacking Midfield": "MID",
        "Right Attacking Midfield": "MID",
        # Forwards/Wingers
        "Left Midfield": "FWD",
        "Right Midfield": "FWD",
        "Left Wing": "FWD",
        "Right Wing": "FWD",
        "Center Forward": "FWD",
        "Left Center Forward": "FWD",
        "Right Center Forward": "FWD",
        "Secondary Striker": "FWD",
    }
    df["position_group"] = df["position"].map(position_map).fillna("MID")

    # FILL REMAINING NaNs INTRODUCED BY RATIO ENGINEERING
    engineered_cols = [
        "pass_completion_u_pressure",
        "progressive_pass_ratio",
        "chance_creation_rate",
        "long_ball_ratio",
        "cross_ratio",
        "progressive_carry_ratio",
        "carry_box_ratio",
        "shot_accuracy",
        "xg_per_90",
        "xg_overperformance",
        "goal_conversion",
        "defensive_actions_p90",
        "tackle_success_proxy",
        "pressure_regain_proxy",
        "dribble_success_rate",
        "duel_win_rate",
        "turnover_rate",
        "touches_per_minute",
    ]
    df[engineered_cols] = df[engineered_cols].fillna(0)

    return df

In [5]:
player_match_logs = preprocess_and_engineer(df)

In [6]:
def aggregate_player_profiles(df, min_minutes=90):
    """
    Aggregates preprocessed per-match logs into per-player profiles.
    categorical: mode
    rates & averages: mean 
    count columns: summed, then expressed as per-90
    engineered ratios: mean
    composite scores: summed, then per-90
    """

    bio_cols = [
        "team",
        "position",
        "position_group",
        "birth_date",
        "nationality",
        "height",
        "weight",
        "jersey_number",
    ]

    rate_cols = [
        "pass_completion",
        "avg_pass_length",
        "avg_carry_dist",
        "avg_xg_per_shot",
        "avg_shot_distance",
        "defensive_height",
        "minutes_played",
        "pass_completion_u_pressure",
        "progressive_pass_ratio",
        "chance_creation_rate",
        "long_ball_ratio",
        "cross_ratio",
        "progressive_carry_ratio",
        "carry_box_ratio",
        "shot_accuracy",
        "xg_overperformance",
        "goal_conversion",
        "tackle_success_proxy",
        "pressure_regain_proxy",
        "dribble_success_rate",
        "duel_win_rate",
        "turnover_rate",
        "touches_per_minute",
        "xg_per_90",
        "defensive_actions_p90",
    ]

    id_cols = ["player_id", "player", "match_id"]
    exclude = set(id_cols + bio_cols + rate_cols)
    num_cols = df.select_dtypes(include=[np.number]).columns.tolist()
    count_cols = [c for c in num_cols if c not in exclude]

    bio_agg = (
        df.groupby(["player_id", "player"])[bio_cols]
        .agg(lambda x: x.mode().iloc[0] if not x.mode().empty else np.nan)
        .reset_index()
    )

    rate_agg = (
        df.groupby(["player_id", "player"])[rate_cols]
        .mean()
        .reset_index()
        .rename(columns={"minutes_played": "avg_minutes_per_match"})
    )

    count_agg = (
        df.groupby(["player_id", "player"])[count_cols + ["minutes_played"]]
        .sum()
        .reset_index()
        .rename(columns={"minutes_played": "total_minutes"})
    )

    profiles = bio_agg.merge(rate_agg, on=["player_id", "player"]).merge(
        count_agg, on=["player_id", "player"]
    )

    profiles = profiles[profiles["total_minutes"] >= min_minutes].copy()

    # Per-90 normalization
    for col in count_cols:
        profiles[f"{col}_p90"] = profiles[col] / (profiles["total_minutes"] / 90)

    match_counts = (
        df.groupby(["player_id", "player"])["match_id"]
        .nunique()
        .reset_index()
        .rename(columns={"match_id": "matches_played"})
    )
    profiles = profiles.merge(match_counts, on=["player_id", "player"])

    return profiles.fillna(0)

In [7]:
final_profiles = aggregate_player_profiles(player_match_logs, min_minutes=90)
print(f"Final dataset shape: {final_profiles.shape}")
print(f"Players included:    {final_profiles['player_id'].nunique()}")
final_profiles.head()

Final dataset shape: (8386, 185)
Players included:    8377


,player_id,player,team,position,position_group,birth_date,nationality,height,weight,jersey_number,...,penalties_conceded_p90,fouls_won_p90,penalties_won_p90,miscontrols_p90,dispossessed_p90,turnovers_p90,attacking_score_p90,defensive_score_p90,pressing_score_p90,matches_played
0,2935.0,Nordi Mukiele Mulere,Paris Saint-Germain,Right Center Back,DEF,0.0,France,0.0,0.0,26,...,0.0,0.641838,0.0,0.356577,0.713154,1.069731,1.684924,9.164025,13.621236,15
1,2936.0,Christophe Kerbrat,Guingamp,Right Center Back,DEF,0.0,France,0.0,0.0,29,...,0.0,0.733126,0.0,0.209465,0.104732,0.314197,0.572647,18.572537,17.525213,29
2,2937.0,Christ-Emmanuel Faitout Maouassa,Montpellier,Left Wing Back,DEF,0.0,France,0.0,0.0,27,...,0.0,2.352941,0.0,1.176471,0.588235,1.764706,1.176471,10.294118,24.705882,2
3,2940.0,Jean Eudès Aholou,Strasbourg,Right Center Midfield,MID,0.0,Côte d'Ivoire,0.0,0.0,6,...,0.0,1.000000,0.0,0.000000,0.000000,0.000000,0.000000,6.000000,7.000000,1
4,2941.0,Ismaïla Sarr,Senegal,Right Wing,FWD,0.0,Senegal,0.0,0.0,18,...,0.0,3.276231,0.0,2.408994,1.734475,4.143469,6.719452,7.226981,23.704497,11


In [12]:
def clean_and_select_features(profiles, zero_threshold=0.95, verbose=True):
    """
    Cleans the aggregated player profile DataFrame:
    1. Drops columns that are 100% zero
    2. Drops columns above `zero_threshold`
    3. Groups sparse-but-valid rare-event columns into composite bins
    4. Fixes broken engineered features
    5. Deduplicates redundant per-90 / raw pairs — keeps per-90 only for
       volume stats, keeps raw only for rates/ratios
    6. Separates GK vs outfield profiles
    Parameters
    """

    df = profiles.copy()
    n_players = len(df)

    # drop columns
    unavailable_bio = ["height", "weight", "birth_date", "jersey_number"]

    # GK-specific columns
    gk_cols = [
        "gk_total_actions",
        "gk_saves",
        "gk_claims",
        "gk_punches",
        "gk_shot_faced",
        "gk_total_actions_p90",
        "gk_saves_p90",
        "gk_claims_p90",
        "gk_punches_p90",
        "gk_shot_faced_p90",
    ]

    # Fully zero features
    broken_features = [
        "tackle_success_proxy",
        "red_cards",
        "red_cards_p90",
        "penalties_won",
        "penalties_won_p90",
        "block_save_blocks",
        "block_save_blocks_p90",
        "dribble_no_touch",
        "dribble_no_touch_p90",
        "shots_open_goal",
        "shots_open_goal_p90",
    ]

    hard_drop = set(unavailable_bio + gk_cols + broken_features)
    hard_drop = [c for c in hard_drop if c in df.columns]
    df.drop(columns=hard_drop, inplace=True)
  

    # Drop columns above sparsity threshold

    num_cols = df.select_dtypes(include=[np.number]).columns.tolist()
    zero_pct = (df[num_cols] == 0).mean()
    too_sparse = zero_pct[zero_pct > zero_threshold].index.tolist()

    # These will be grouped
    will_group = {
        "inswinging_crosses",
        "outswinging_crosses",
        "cut_backs",
        "shots_first_time",
        "shots_one_on_one",
        "block_deflections",
        "block_offensive",
        "nutmegs",
        "through_balls",
        "goal_kick_passes",
        "throw_in_passes",
        "total_50_50s",
        "50_50s_won",
        "penalties_conceded",
        "corner_passes",
        "free_kick_passes",
    }
    # Also keep p90 counterparts
    will_group_p90 = {c + "_p90" for c in will_group}
    keep_sparse = will_group | will_group_p90

    sparse_to_drop = [c for c in too_sparse if c not in keep_sparse]
    df.drop(columns=sparse_to_drop, inplace=True)

    # Group sparse-but-valid

    def _get(col):
        """Return p90 col if available, else raw, else zeros."""
        p90 = col + "_p90"
        if p90 in df.columns:
            return df[p90]
        if col in df.columns:
            return df[col]
        return pd.Series(0, index=df.index)

    # Cross creativity
    df["cross_quality_index"] = (
        _get("inswinging_crosses") * 1.2
        + _get("outswinging_crosses") * 1.2
        + _get("cut_backs") * 1.5
    )

    # Shot quality
    df["shot_quality_index"] = (
        _get("shots_first_time") * 1.5 + _get("shots_one_on_one") * 2.0
    )

    # Defensive block quality
    df["block_quality_index"] = (
        _get("block_deflections") * 1.0
        + _get("block_offensive") * 1.5
    )

    # Pass creativity
    df["pass_creativity_index"] = (
        _get("through_balls") * 2.0
        + _get("key_passes") * 1.0
        + _get("goal_assists") * 1.5
    )

    # Set piece involvement
    df["set_piece_delivery_index"] = (
        _get("corner_passes") * 1.0
        + _get("free_kick_passes") * 1.0
        + _get("throw_in_passes") * 0.5
        + _get("goal_kick_passes") * 0.5
    )

    # Dribble composite
    df["dribble_composite"] = _get("successful_dribbles") * 1.5 + _get("nutmegs") * 2.0

    # Duel composite
    df["duel_composite"] = (
        _get("total_50_50s") * 0.5
        + _get("50_50s_won") * 1.5
        + _get("aerial_duels") * 1.0
        + _get("ground_duels") * 1.0
        + _get("duels_won") * 1.5
    )

    # Discipline score
    df["discipline_score"] = (
        _get("fouls_committed") * 1.0
        + _get("yellow_cards") * 3.0
        + _get("penalties_conceded") * 5.0
        + _get("turnovers") * 0.5
    )

    # Drop the sparse
    composite_inputs = (
        will_group
        | will_group_p90
        | {
            "aerial_duels",
            "aerial_duels_p90",
            "ground_duels",
            "ground_duels_p90",
            "duels_won",
            "duels_won_p90",
            "yellow_cards",
            "yellow_cards_p90",
            "fouls_committed",
            "fouls_committed_p90",
            "turnovers",
            "turnovers_p90",
        }
    )
    drop_after_composite = [c for c in composite_inputs if c in df.columns]
    df.drop(columns=drop_after_composite, inplace=True)

    # Fix broken engineered features

    # (tackles / total_duels) is wrong the correct is (tackles / (tackles + total_dribbles_faced_proxy))
    if "tackles" in df.columns:
        contested = df.get("tackles", 0) + df.get("dispossessed", 0)
        df["tackle_win_rate"] = np.where(
            contested > 0, df["tackles"] / contested, np.nan
        ).astype(float)
        df["tackle_win_rate"] = df["tackle_win_rate"].fillna(0)
    else:
        df["tackle_win_rate"] = 0

    # pressure_regain_proxy
    if "pressure_regain_proxy" in df.columns:
        df["pressure_regain_proxy"] = df["pressure_regain_proxy"].fillna(0)

    # Remove raw count duplicates
    # for any (raw_col, raw_col_p90) pair, keep p90 only.
    # Exception: keep raw totals for 'total_minutes', 'matches_played', career-level accumulators used in ratio computation.

    keep_raw_always = {
        "total_minutes",
        "matches_played",
        "player_id",
        "match_id",
        "total_passes",
        "total_shots",
        "total_dribbles",
        "total_carries",
        "total_duels",
        "total_defensive_actions",
        "touches",
        "goals",
        "total_xg",
    }

    raw_with_p90 = [
        c for c in df.columns if (c + "_p90") in df.columns and c not in keep_raw_always
    ]
    df.drop(columns=raw_with_p90, inplace=True)

    # GK

    outfield_mask = df["position_group"] != "GK"
    df_outfield = df[outfield_mask].copy()
    df_gk = df[~outfield_mask].copy()

    num_out = df_outfield.select_dtypes(include=[np.number]).columns
    all_zero_out = num_out[(df_outfield[num_out] == 0).all()]
    df_outfield.drop(columns=all_zero_out, inplace=True)

    return df_outfield, df_gk

In [14]:
outfield_profiles, gk_profiles = clean_and_select_features(
    final_profiles, zero_threshold=0.95, verbose=True
)
outfield_profiles.info()

<class 'pandas.DataFrame'>
Index: 7744 entries, 0 to 8385
Data columns (total 90 columns):
 #   Column                        Non-Null Count  Dtype  
---  ------                        --------------  -----  
 0   player_id                     7744 non-null   float64
 1   player                        7744 non-null   str    
 2   team                          7744 non-null   str    
 3   position                      7744 non-null   str    
 4   position_group                7744 non-null   str    
 5   nationality                   7744 non-null   str    
 6   pass_completion               7744 non-null   float64
 7   avg_pass_length               7744 non-null   float64
 8   avg_carry_dist                7744 non-null   float64
 9   avg_xg_per_shot               7744 non-null   float64
 10  avg_shot_distance             7744 non-null   float64
 11  defensive_height              7744 non-null   float64
 12  avg_minutes_per_match         7744 non-null   float64
 13  pass_completion_u_p

In [24]:
from sklearn.preprocessing import RobustScaler

def rationalize_and_scale(df, verbose=True):
    """
    1. Drop pure metadata / leakage columns
    2. Encode position_group as ordinal dummies
    3. Scale numeric features with RobustScaler (median/IQR — resistant to
       the outliers)
    """

    df = df.copy()

    # Separate metadata from features
    meta_cols = [
        "player_id",
        "player",
        "team",
        "position",
        "position_group",
        "nationality",
        "total_minutes",
        "matches_played",
        "avg_minutes_per_match",
    ]
    meta_cols = [c for c in meta_cols if c in df.columns]
    df_meta = df[meta_cols].copy()
    df_feat = df.drop(columns=meta_cols)

    #  Encode position_group

    pos_dummies = pd.get_dummies(
        df_meta["position_group"],
        prefix="pos",
        drop_first=True
    ).astype(int)

    df_feat = pd.concat([df_feat, pos_dummies], axis=1)

    # Final NaN
    df_feat.replace([np.inf, -np.inf], np.nan, inplace=True)

    for col in df_feat.select_dtypes(include=[np.number]).columns:
        median_val = df_feat[col].median()
        df_feat[col] = df_feat[col].fillna(median_val)

    # RobustScaler

    dummy_cols = [c for c in df_feat.columns if c.startswith("pos_")]
    scale_cols = [c for c in df_feat.columns if c not in dummy_cols]

    scaler = RobustScaler()
    scaled_vals = scaler.fit_transform(df_feat[scale_cols])
    df_scaled = pd.DataFrame(scaled_vals, columns=scale_cols, index=df_feat.index)

    # Re-attach unscaled dummies
    df_model = pd.concat([df_scaled, df_feat[dummy_cols]], axis=1)

    feature_cols = df_model.columns.tolist()

    if verbose:
        print(f"RobustScaler applied to {len(scale_cols)} columns")
        print(f"Model-ready shape: {df_model.shape}")
        print(f"Features: {len(feature_cols)}")
        print(f"Meta rows: {df_meta.shape}")

    return df_model, df_meta, scaler, feature_cols

In [26]:
df_model, df_meta, scaler, feature_cols = rationalize_and_scale(
    outfield_profiles, verbose=True
)

RobustScaler applied to 81 columns
Model-ready shape: (7744, 83)
Features: 83
Meta rows: (7744, 9)


In [27]:
df_model[feature_cols[:10]].describe().round(3).T[["mean", "50%", "std"]]

,mean,50%,std
pass_completion,-0.112,0.0,0.979
avg_pass_length,0.028,0.0,0.868
avg_carry_dist,0.116,-0.0,1.022
avg_xg_per_shot,0.317,-0.0,1.049
avg_shot_distance,0.072,-0.0,0.636
defensive_height,0.008,-0.0,0.658
pass_completion_u_pressure,-0.302,0.0,0.977
progressive_pass_ratio,0.016,0.0,0.772
chance_creation_rate,0.323,0.0,1.218
long_ball_ratio,0.081,-0.0,0.856


In [32]:
def get_interpretable_profiles(df_meta, df_model, scaler, feature_cols):
    """
    Returns unscaled feature values joined with metadata.
    """
    dummy_cols = [c for c in feature_cols if c.startswith("pos_")]
    scale_cols = [c for c in feature_cols if c not in dummy_cols]

    unscaled_vals = scaler.inverse_transform(df_model[scale_cols])
    df_unscaled = pd.DataFrame(unscaled_vals, columns=scale_cols, index=df_model.index)
    df_unscaled = pd.concat([df_unscaled, df_model[dummy_cols]], axis=1)

    return pd.concat(
        [df_meta.reset_index(drop=True), df_unscaled.reset_index(drop=True)], axis=1
    )

In [ ]:
interpretable_profiles = get_interpretable_profiles(
    df_meta, df_model, scaler, feature_cols
)
interpretable_profiles.shape

(7744, 92)

In [36]:
outfield_profiles.to_csv("profiles_cleand.csv", index=False)
df_model.to_csv("profiles_scaled.csv", index=False)
interpretable_profiles.to_csv("interpretable_profiles.csv", index=False)